# Final Project Starting Guide

Hello everyone, welcome to the final project! This notebook is provided to you to reiterate the rules and guidelines, and give you some starting points.

### What we provide

In this project, we will provide you with 
- This starting guide
- A working API that you can access under ASU network (i.e., on campus or with VPN)
- A starting development data that you can use to develop your agent. It contains 1,000 instances with {domain, input, expected_output}

### Your goal

In this project, you will implement an inference-time agent to solve reasoning requests, as those provided in the development data. The grading of this project will be effort-based and you will get full credit if you produce the minimum deliverables below, with subject to the rules and requirements below.

#### Minimum Deliverables

1. A working agent loop (in the form of a Github project) that the TA can run, and implements *at least eight* inference-time algorithms or techniques.
2. Outputs from your agent on the released test data (see important dates). 
3. A short one-page report on how your agent works, and pointer to important techniques (referece to code blocks).

#### Rules and Requirements
1. You must only use our provided API call to access LLMs; meaning that you cannot use any other LLMs in any other way within your agent loop. Some exceptions may be made if you call certain external tools (e.g., Google search) that use some LLMs internally. Please discuss any exceptions with us to avoid penalties up to 50% of the project grade.
2. You must not hardcode a full delegation to an external tool (e.g., google_search(input_problem)). Such delegations must be automatically selected/decided by your agent. Hardcode delegations will lead to a zero.
3. You cannot use Cursor or any AI coding aids to implement the final project. You can, however, ask LLMs (or other online resources) for conceptual clarification or code examples. Your final project should not contain any blocks of code (i.e., > 3 lines) that are written by AI. Violations will lead to a zero.
4. Your agent should be able to run efficiently, with <20 LLM calls per question. Exceptions may be made when you have a complicated agent but please discuss with us. Up to 10% of the project grade may be deducted if we observe very inefficient LLM usages that do not clearly benefit the performance.
5. Your agent must run without any requests to any paid services (paid is defined by if the TA has to pay to run it, regardless of whether you actuallly pay for it or not.) Violations will lead to a zero.
6. You must submit a Github project link as your code submission. All changes must be tracked and any commits should be within 100 lines of +/- with good messages. Points will be deducted to up to 25% of the project grade if we observe "magic commits" or too few commits. 


### Suggestions
1. Start early, please.
2. You should consider how you can evaluate whether your output is good enough compared to the provided expected_outputs, and we will not release how we will actually evaluate your outputs; meaning that you have to try to predict how we will evaluate things.
3. Start with a basic implementation, and iterate based on mistakes/feedbacks.
4. Find more development data, or create your own cases to stree-test your agent. 
5. You are free to modify any provided code in this starting guide, or not using any of these code at all.

### Important dates
- **Release of final test data**: 04/07/2026
- **Deadline for submitting all deliverables**: 04/23/2026

### Extra Credit. 
The top 20 projects (ranked by performance metrics on the test data and at the TA's discretion of implementation quality) will be given extra credits. The actual credits will be between 1% to 7.5% depending on the ranking.

In [2]:
# %% Minimal setup
# If needed (uncomment in a notebook):
# !pip install requests python-dotenv
from dotenv import load_dotenv
load_dotenv()
import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "sk-BxBQNNMRbPJDHf4G2ftsiA")
API_BASE = os.getenv("API_BASE", "https://openai.rc.asu.edu/v1")  
MODEL    = os.getenv("MODEL_NAME", "qwen3-30b-a3b-instruct-2507")              

def call_model_chat_completions(prompt: str,
                                system: str = "You are a helpful assistant. Reply with only the final answer-no explanation.",
                                model: str = MODEL,
                                temperature: float = 0.0,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 128,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}


## 1) Smoke test: direct inference

We’ll do a single request with a strict instruction to answer briefly.  
*If you see an auth error, set `OPENAI_API_KEY` and (if needed) `API_BASE`/`MODEL_NAME`.*


In [3]:
# %% Direct call example
demo_prompt = "What is 17 + 28? Answer with just the number."
result = call_model_chat_completions(demo_prompt)
print("OK:", result["ok"], "HTTP:", result["status"])
print("MODEL SAYS:", (result["text"] or "").strip())

# Optional: Inspect rate-limit headers if your provider exposes them
for k in ["x-ratelimit-remaining-requests", "x-ratelimit-limit-requests", "x-request-id"]:
    if k in result["headers"]:
        print(f"{k}: {result['headers'][k]}")


OK: True HTTP: 200
MODEL SAYS: 45


## 2) A tiny test set (3 questions)

We’ll cover:
1. **Math reasoning** - inequality solving,
2. **Common sense** - buoyancy/ice & water,
3. **Logic** - a classic race-position puzzle.

We also tightly constrain the required answer forms to enable simple auto‑grading.


In [4]:
# %% Define three tests: input + expected
tests = [
    {
        "id": "math_inequality",
        "type": "numeric",  # grader will prefer numeric extraction
        "prompt": "Solve for the smallest integer n such that 3n + 5 > 26. Answer with just the integer.",
        "expected": "8",    # Because 3n > 21 => n > 7, smallest integer is 8
    },
    {
        "id": "commonsense_ice",
        "type": "text",
        "prompt": (
            "You place an ice cube in a glass of water and mark the water level. "
            "After the ice melts, does the water level rise, fall, or stay the same? "
            "Answer with exactly one of: 'rise', 'fall', 'stay the same'."
        ),
        "expected": "stay the same",
    },
    {
        "id": "logic_race",
        "type": "text",
        "prompt": (
            "In a race, you pass the person in second place. What position are you now in? "
            "Answer with a single word like 'first', 'second', 'third'."
        ),
        "expected": "second",
    },
]


## 3) Minimal evaluator

We provide some example code to decide whether the agent outputs match the expected outputs, just to give you an idea of how evaluations can be done. You are free to use this code, or not.

In [5]:
# %% Simple normalization and evaluation helpers
def normalize_text(s: str) -> str:
    s = (s or "").strip().lower()
    # Remove surrounding punctuation and extra whitespace
    s = re.sub(r"[^\w\s\-']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Map common synonyms used in these tests
    synonyms = {
        "unchanged": "stay the same",
        "no change": "stay the same",
        "same": "stay the same",
        "second place": "second",
        "2nd": "second",
        "first place": "first",
        "third place": "third",
    }
    return synonyms.get(s, s)

def extract_number(s: str):
    # Returns first number occurrence as string if found, else None
    if not s:
        return None
    m = re.search(r"[-+]?\d+(\.\d+)?", s)
    return m.group(0) if m else None

def grade(expected: str, got: str, kind: str) -> bool:
    if kind == "numeric":
        exp_num = extract_number(expected)
        got_num = extract_number(got)
        return (exp_num is not None) and (got_num == exp_num)
    else:
        return normalize_text(got) == normalize_text(expected)

def evaluate_tests(tests, model=MODEL):
    rows = []
    for t in tests:
        r = call_model_chat_completions(
            t["prompt"],
            system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
            model=model,
            temperature=0.0,
        )
        got = (r["text"] or "").strip()
        is_correct = grade(t["expected"], got, t["type"])
        rows.append({
            "id": t["id"],
            "expected": t["expected"],
            "got": got,
            "correct": is_correct,
            "status": r["status"],
            "error": r["error"],
        })
        # Tiny pacing to be polite to the API
        time.sleep(0.2)

    # Print a small report
    correct = sum(1 for x in rows if x["correct"])
    print(f"Score: {correct}/{len(rows)} correct")
    for x in rows:
        mark = "✅" if x["correct"] else "❌"
        print(f"{mark} {x['id']}: expected={x['expected']!r}, got={x['got']!r} (HTTP {x['status']})")
        if x["error"]:
            print("   error:", x["error"])
    return rows

results = evaluate_tests(tests)


Score: 2/3 correct
❌ math_inequality: expected='8', got='21' (HTTP 200)
✅ commonsense_ice: expected='stay the same', got='stay the same' (HTTP 200)
✅ logic_race: expected='second', got='second' (HTTP 200)


In [6]:
def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)


In [8]:
def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        # 1) Get model prediction
        r = call_model_chat_completions(
            t["prompt"],
            system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
            model=model,
            temperature=0.0,
        )
        got = (r.get("text") or "").strip()

        # 2) LLM-as-a-judge: strict True/False
        is_correct = self_evaluate(
            question=t["prompt"],
            prediction=got,
            expected_answer=t["expected"],
            model=judge_model,
        )

        row = {
            "id": t.get("id", "<unnamed>"),
            "expected": t["expected"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: expected={row['expected']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows

# Example:
results_llm_judge = self_evaluate_tests(tests, verbose=True, model=MODEL, grader_model=MODEL)


❌ math_inequality: expected='8', got='21' (HTTP 200)
✅ commonsense_ice: expected='stay the same', got='stay the same' (HTTP 200)
✅ logic_race: expected='second', got='second' (HTTP 200)


## 4) Call-budget tracker

The project rule is a 20-call-per-question hard limit. `CallBudget` keeps count and `can_afford(n)` lets each technique decide whether to run an expensive plan or fall back to the cheap CoT baseline. We monkey-patch `call_model_chat_completions` so every request bumps the counter automatically, instead of threading a counter through every function.


In [9]:
class CallBudget:
    """Tracks LLM calls against a per-question hard limit.

    We bump() on every request, reset() at the start of each question, and
    techniques call can_afford(n) before committing to an expensive plan. If the
    budget is too tight, the technique falls back to cot() instead.
    """

    def __init__(self, hard_limit=20):
        self.hard_limit = hard_limit
        self.used = 0

    def reset(self):
        self.used = 0

    def bump(self, n=1):
        self.used += n

    @property
    def remaining(self):
        return max(0, self.hard_limit - self.used)

    def can_afford(self, n):
        return (self.used + n) <= self.hard_limit


# Global tracker used by every technique and by the monkey-patched API call.
CALLS = CallBudget(hard_limit=20)


In [10]:
# Monkey-patch call_model_chat_completions so every request bumps CALLS and so
# techniques can opt into a larger max_tokens. We keep a reference to the
# original so this cell is re-runnable without stacking wrappers.
if "_original_call_model_chat_completions" not in dir():
    _original_call_model_chat_completions = call_model_chat_completions


def call_model_chat_completions(prompt,
                                system="You are a helpful assistant. Reply with only the final answer, no explanation.",
                                model=MODEL,
                                temperature=0.0,
                                timeout=60,
                                max_tokens=128):
    """Wrapper over the original API call.

    Adds two things: (1) bumps the global CALLS counter on every request and
    (2) accepts max_tokens so techniques that need long reasoning traces have
    room. The endpoint, key, and model are unchanged -- same API, just with a
    counter around it.
    """
    CALLS.bump()

    url = API_BASE + "/chat/completions"
    headers = {
        "Authorization": "Bearer " + API_KEY,
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        # Non-200: try to surface the error body as text for debugging.
        err_text = None
        try:
            err_text = resp.json()
        except Exception:
            err_text = resp.text
        return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}


def ask(prompt, system=None, temperature=0.0, max_tokens=512):
    """Thin convenience wrapper: call the model and return just the text string.

    Returns '' on API failure so callers can safely .strip()/.lower() without
    worrying about None.
    """
    sys_msg = system if system is not None else (
        "You are a careful reasoner. Think clearly and give only what is asked."
    )
    r = call_model_chat_completions(
        prompt,
        system=sys_msg,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    if not r.get("ok"):
        return ""
    return (r.get("text") or "").strip()


# Quick sanity check: should consume 1 call and leave 19 of the 20-call budget.
CALLS.reset()
_sanity = ask("Reply with just OK.", max_tokens=8)
print("sanity reply:", repr(_sanity), " used=", CALLS.used, " remaining=", CALLS.remaining)


sanity reply: 'OK'  used= 1  remaining= 19


## 5) Dev data loader

Loads the 1,000-instance development set and prints the domain distribution so we can sanity-check that the router covers every domain we actually see.


In [11]:
# Look in a couple of likely locations -- the notebook can live alongside the data
# or one folder up.
_candidate_paths = [
    "cse476_final_project_dev_data.json",
    os.path.join("final_project_tutorial_and_dev_data", "cse476_final_project_dev_data.json"),
    os.path.join("..", "cse476_final_project_dev_data.json"),
]

DEV_DATA = None
for path in _candidate_paths:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            DEV_DATA = json.load(f)
        print("Loaded " + str(len(DEV_DATA)) + " instances from " + path)
        break

if DEV_DATA is None:
    raise FileNotFoundError(
        "cse476_final_project_dev_data.json not found. Tried: "
        + ", ".join(_candidate_paths)
    )

# Domain distribution: a sanity-check that the routing policy covers every domain
# we actually see.
domain_counts = {}
for inst in DEV_DATA:
    d = inst.get("domain", "<missing>")
    domain_counts[d] = domain_counts.get(d, 0) + 1

print("\nDomain distribution:")
for d, c in sorted(domain_counts.items(), key=lambda kv: -kv[1]):
    print("  " + d.rjust(20) + ": " + str(c))


Loaded 1000 instances from cse476_final_project_dev_data.json

Domain distribution:
          common_sense: 400
                  math: 300
                coding: 100
     future_prediction: 100
              planning: 100


## 6) Answer extractor

Techniques tend to emit verbose traces. `extract_final_answer` pulls out the final answer in this priority order: `Final answer: X`, then `\boxed{X}` (LaTeX convention used throughout the dev set), then the last non-empty line as a fallback. `FINAL_ANSWER_INSTRUCTION` is appended to technique prompts so the model emits something parseable.


In [12]:
FINAL_ANSWER_INSTRUCTION = (
    "\n\nWhen you are done, write your final answer on its own line in this "
    "exact format:\n"
    "Final answer: <your answer>\n"
    "Do not add anything after that line."
)

# Compiled once. Matches 'Final answer: X' or 'final answer is X', capturing the
# answer up to end of line.
_FINAL_ANSWER_RE = re.compile(
    r"final\s*answer\s*(?:is|:)\s*(.+?)(?:\n|$)",
    re.IGNORECASE,
)

# Simple \boxed{...} matcher. We don't handle nested braces -- the dev set only
# uses single-level boxed answers in practice.
_BOXED_RE = re.compile(r"\\boxed\{([^{}]*)\}")


def extract_final_answer(text):
    """Parse the final answer out of a model reply.

    Priority: (1) 'Final answer: X', (2) \\boxed{X}, (3) the last non-empty line.
    Returns a stripped string, or '' if the input is empty.
    """
    if not text:
        return ""

    # 1) Final answer: X
    m = _FINAL_ANSWER_RE.search(text)
    if m:
        answer = m.group(1).strip()
        answer = answer.strip(" .\"'`")
        if answer:
            return answer

    # 2) \boxed{X} -- prefer the last one if there are several.
    boxed_matches = _BOXED_RE.findall(text)
    if boxed_matches:
        return boxed_matches[-1].strip()

    # 3) Last non-empty line.
    for line in reversed(text.splitlines()):
        line = line.strip()
        if line:
            return line.strip(" .\"'`")

    return ""


## 7) Safe Python execution tool

A sandbox for the `react` and `tool_augmented` techniques. Short snippets only. A denylist blocks obviously-dangerous patterns and a restricted builtins dict provides only a handful of safe functions plus the `math` module. We try `eval` first (so `1 + 2` works without `print`), then fall back to `exec` while capturing stdout.


In [14]:
import io
import contextlib

# Patterns the model's snippet is not allowed to contain. A simple substring
# check, not a tokenizer -- its purpose is a hard-fail early warning, not an
# airtight sandbox. The restricted builtins dict below is the second line of defense.
_FORBIDDEN_PATTERNS = [
    "import os",
    "import sys",
    "open(",
    "__import__",
    "subprocess",
    "socket",
    "shutil",
]


def _safe_import(name, globals=None, locals=None, fromlist=(), level=0):
    """Whitelisted import used inside python_exec. Allows a handful of safe
    stdlib modules so that snippets like 'import math' can succeed."""
    _allowed = {
        "math", "fractions", "itertools", "collections",
        "statistics", "functools", "re", "decimal",
    }
    if name not in _allowed:
        raise ImportError("import of '" + name + "' is not allowed in python_exec")
    return __import__(name, globals, locals, fromlist, level)


def python_exec(code_str, timeout=3):
    """Run a short Python snippet and return stdout, or an ERROR string.

    The `timeout` argument is advisory -- Python's eval/exec can't be preempted
    portably, so we rely on the denylist and short snippets to stay in bounds.
    """
    if not code_str or not code_str.strip():
        return "ERROR: ValueError: empty code"

    for pattern in _FORBIDDEN_PATTERNS:
        if pattern in code_str:
            return "ERROR: SecurityError: forbidden pattern " + repr(pattern)

    # Restricted globals: a handful of builtins plus the math module preloaded.
    import math as _math
    restricted_globals = {
        "__builtins__": {
            "print": print,
            "len": len, "range": range, "abs": abs, "min": min, "max": max,
            "sum": sum, "round": round, "int": int, "float": float, "str": str,
            "list": list, "tuple": tuple, "dict": dict, "set": set,
            "enumerate": enumerate, "zip": zip, "sorted": sorted,
            "reversed": reversed, "any": any, "all": all, "map": map,
            "filter": filter, "pow": pow, "divmod": divmod,
            "True": True, "False": False, "None": None,
            "__import__": _safe_import,
            "isinstance": isinstance, "type": type,
        },
        "math": _math,
    }
    local_vars = {}

    stdout_capture = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_capture):
            # Try eval first so 'print(2**10)' AND '2**10' both work.
            try:
                value = eval(code_str, restricted_globals, local_vars)
                if value is not None:
                    print(repr(value))
            except SyntaxError:
                # Not a single expression -- treat the whole thing as a block.
                exec(code_str, restricted_globals, local_vars)
    except Exception as e:
        return "ERROR: " + type(e).__name__ + ": " + str(e)

    out = stdout_capture.getvalue().strip()
    return out if out else ""


## 8) Eight inference-time techniques

Each technique takes a question string and returns a final-answer string. Each one starts with `CALLS.can_afford(n)` and falls back to `cot()` if the budget can't cover the worst case. The functions are ordered cheap-to-expensive so dependencies go top-down: `cot` first, then multi-call variants.


In [15]:
def cot(question):
    """Zero-shot chain-of-thought. The cheapest baseline: one call."""
    if not CALLS.can_afford(1):
        # We're already out of budget. Return empty so the caller can decide.
        return ""

    prompt = (
        "Solve this problem step by step. Show your reasoning briefly, then "
        "give the final answer.\n\nProblem:\n"
        + question
        + FINAL_ANSWER_INSTRUCTION
    )
    reply = ask(prompt, temperature=0.0, max_tokens=512)
    return extract_final_answer(reply)


In [16]:
def self_consistency(question, n_samples=5):
    """Sample n CoT traces at temperature 0.7 and majority-vote the answers.

    Good for questions with a small, discrete answer space (common-sense, short
    numeric answers). Votes are tallied on lowercased/stripped answers.
    """
    if not CALLS.can_afford(n_samples):
        return cot(question)

    prompt = (
        "Solve this problem step by step. Show your reasoning briefly, then "
        "give the final answer.\n\nProblem:\n"
        + question
        + FINAL_ANSWER_INSTRUCTION
    )

    votes = {}      # normalized -> count
    display = {}    # normalized -> original-cased answer to return
    for _ in range(n_samples):
        reply = ask(prompt, temperature=0.7, max_tokens=512)
        ans = extract_final_answer(reply)
        if not ans:
            continue
        key = ans.lower().strip()
        votes[key] = votes.get(key, 0) + 1
        # Keep the first occurrence's casing so we return a readable string.
        if key not in display:
            display[key] = ans

    if not votes:
        return cot(question)

    winner_key = max(votes.items(), key=lambda kv: kv[1])[0]
    return display[winner_key]


In [17]:
def tree_of_thoughts(question, breadth=3, depth=2):
    """Generate `breadth` first-thoughts, score each, then expand the best one.

    Budget: breadth (candidates) + breadth (scores) + depth (expansion) + 1 (final)
    = about 6-10 calls at defaults.
    """
    needed = breadth + breadth + depth + 1
    if not CALLS.can_afford(needed):
        return cot(question)

    # Step 1: generate distinct first thoughts at higher temperature for diversity.
    candidate_prompt = (
        "Here is a problem. Propose ONE distinct first step toward solving it. "
        "Be concrete. Do not give the final answer yet. Write 2-3 sentences.\n\n"
        "Problem:\n" + question
    )
    candidates = []
    for _ in range(breadth):
        c = ask(candidate_prompt, temperature=0.8, max_tokens=256)
        if c:
            candidates.append(c)
    if not candidates:
        return cot(question)

    # Step 2: score each candidate on a 1-10 scale in a separate call.
    scored = []  # list of (score_int, candidate_text)
    for c in candidates:
        score_prompt = (
            "Rate how promising this first step is for solving the problem, on "
            "a scale of 1 to 10. Reply with just the integer.\n\n"
            "Problem:\n" + question + "\n\nFirst step:\n" + c
        )
        s_text = ask(score_prompt, temperature=0.0, max_tokens=8)
        m = re.search(r"\d+", s_text)
        score = int(m.group(0)) if m else 0
        scored.append((score, c))

    # Step 3: expand the highest-scored candidate with `depth` more steps.
    scored.sort(key=lambda sc: -sc[0])
    best_first = scored[0][1]

    trace = "First step:\n" + best_first
    for step_idx in range(depth):
        expand_prompt = (
            "Continue reasoning toward the answer. Write the next step only "
            "(2-3 sentences). Do not restate earlier steps.\n\n"
            "Problem:\n" + question + "\n\n" + trace
        )
        nxt = ask(expand_prompt, temperature=0.3, max_tokens=256)
        if not nxt:
            break
        trace += "\n\nStep " + str(step_idx + 2) + ":\n" + nxt

    # Step 4: finalize.
    final_prompt = (
        "Given the reasoning below, state the final answer to the problem.\n\n"
        "Problem:\n" + question + "\n\n" + trace + FINAL_ANSWER_INSTRUCTION
    )
    final_reply = ask(final_prompt, temperature=0.0, max_tokens=256)
    return extract_final_answer(final_reply)


In [18]:
def self_refine(question):
    """Draft, critique, revise. If critique says 'NO ISSUES' we skip the revise."""
    if not CALLS.can_afford(3):
        return cot(question)

    # 1) Draft a solution.
    draft_prompt = (
        "Solve this problem step by step and give the final answer.\n\nProblem:\n"
        + question
        + FINAL_ANSWER_INSTRUCTION
    )
    draft = ask(draft_prompt, temperature=0.2, max_tokens=512)
    draft_answer = extract_final_answer(draft)

    # 2) Critique. We ask for 'NO ISSUES' as a short-circuit so we can save a call
    # when the draft already looks correct.
    critique_prompt = (
        "Critically review the solution below. If the reasoning is correct and "
        "the final answer is right, reply with exactly: NO ISSUES\n"
        "Otherwise, list the specific errors in 2-4 sentences.\n\n"
        "Problem:\n" + question + "\n\nSolution:\n" + draft
    )
    critique = ask(critique_prompt, temperature=0.0, max_tokens=256)
    if "NO ISSUES" in critique.upper():
        return draft_answer

    # 3) Revise using the critique.
    revise_prompt = (
        "Revise the solution to fix the issues in the critique, then state the "
        "corrected final answer.\n\n"
        "Problem:\n" + question + "\n\nPrevious solution:\n" + draft
        + "\n\nCritique:\n" + critique + FINAL_ANSWER_INSTRUCTION
    )
    revised = ask(revise_prompt, temperature=0.0, max_tokens=512)
    return extract_final_answer(revised) or draft_answer


In [19]:
# ReAct action parsing. The model emits one of:
#   Action: python_exec[<code>]
#   Action: finish[<answer>]
# The bracket contents may contain newlines, so we use DOTALL and non-greedy.
_REACT_ACTION_RE = re.compile(
    r"Action:\s*(python_exec|finish)\s*\[(.*?)\](?:\s*$|\s*\n)",
    re.DOTALL | re.IGNORECASE,
)


def react(question, max_steps=5):
    """Thought/Action loop. Each step the model emits one action; we execute it.

    Exits on finish[...] or after max_steps. If we run out of steps without a
    finish, we do one summarization call to pull a final answer from the trace.
    """
    # Budget: 1 call per step + 1 for the summarization fallback.
    if not CALLS.can_afford(max_steps + 1):
        return cot(question)

    system_msg = (
        "You are a reasoning agent that solves problems by thinking step by step "
        "and optionally running Python code. At each turn, write ONE Thought "
        "followed by ONE Action. Actions must match one of:\n"
        "  Action: python_exec[<python code>]\n"
        "  Action: finish[<final answer>]\n"
        "Only the `math` module is available; no os/sys/subprocess imports. "
        "Keep code short."
    )

    trace = "Problem: " + question + "\n"
    last_action_was_finish = False
    final_answer = ""

    for step in range(max_steps):
        prompt = trace + "\nWrite the next Thought and Action."
        reply = ask(prompt, system=system_msg, temperature=0.0, max_tokens=384)
        if not reply:
            break

        trace += "\n" + reply

        m = _REACT_ACTION_RE.search(reply)
        if not m:
            # No parsable action. Nudge the model and continue.
            trace += "\nObservation: (no action parsed; please emit Action: python_exec[...] or Action: finish[...])"
            continue

        action_name = m.group(1).lower()
        action_arg = m.group(2).strip()

        if action_name == "finish":
            final_answer = action_arg
            last_action_was_finish = True
            break

        # python_exec
        obs = python_exec(action_arg)
        if len(obs) > 400:
            obs = obs[:400] + "...(truncated)"
        trace += "\nObservation: " + obs

    if last_action_was_finish:
        return final_answer.strip()

    # Fallback: ask the model to state a final answer from whatever trace we have.
    if CALLS.can_afford(1):
        summarize_prompt = (
            "Based on the trace below, state the final answer to the problem.\n\n"
            + trace + FINAL_ANSWER_INSTRUCTION
        )
        reply = ask(summarize_prompt, temperature=0.0, max_tokens=128)
        return extract_final_answer(reply)
    return ""


In [20]:
def least_to_most(question, max_subquestions=3):
    """Decompose into sub-questions, solve each in order, then synthesize.

    Call count: 1 (decompose) + N (one per sub-question, N <= max_subquestions)
    + 1 (synthesize). Worst case at default settings: 5 calls.
    """
    if not CALLS.can_afford(1 + max_subquestions + 1):
        return cot(question)

    # 1) Decompose into a numbered list.
    decompose_prompt = (
        "Break this problem into at most " + str(max_subquestions) + " simple "
        "sub-questions that, answered in order, would solve it. Write them as a "
        "numbered list. Do NOT answer them yet.\n\nProblem:\n" + question
    )
    decomp = ask(decompose_prompt, temperature=0.0, max_tokens=256)

    # Parse numbered lines. Accept '1.', '1)', or '1:' as the delimiter.
    subqs = []
    for line in decomp.splitlines():
        line = line.strip()
        m = re.match(r"^\d+[\.\):]\s*(.+)$", line)
        if m:
            subqs.append(m.group(1).strip())
    subqs = subqs[:max_subquestions]
    if not subqs:
        return cot(question)

    # 2) Solve each sub-question sequentially, with prior Q/A pairs in context.
    answers = []  # list of (subq, answer) tuples
    for i, sq in enumerate(subqs):
        context_lines = []
        for j, (prev_q, prev_a) in enumerate(answers):
            context_lines.append("Q" + str(j + 1) + ": " + prev_q + "\nA" + str(j + 1) + ": " + prev_a)
        context_block = "\n".join(context_lines)

        solve_prompt = "Original problem:\n" + question
        if context_block:
            solve_prompt += "\n\nSolved so far:\n" + context_block
        solve_prompt += "\n\nNow answer sub-question " + str(i + 1) + ":\n" + sq
        solve_prompt += "\n\nReply with just the answer, briefly."

        a = ask(solve_prompt, temperature=0.0, max_tokens=256)
        answers.append((sq, a.strip()))

    # 3) Synthesize the final answer from the sub-answers.
    context_lines = []
    for j, (prev_q, prev_a) in enumerate(answers):
        context_lines.append("Q" + str(j + 1) + ": " + prev_q + "\nA" + str(j + 1) + ": " + prev_a)
    synth_prompt = (
        "Original problem:\n" + question
        + "\n\nSub-question answers:\n" + "\n".join(context_lines)
        + "\n\nUsing the above, give the final answer to the original problem."
        + FINAL_ANSWER_INSTRUCTION
    )
    final_reply = ask(synth_prompt, temperature=0.0, max_tokens=256)
    return extract_final_answer(final_reply)


In [21]:
# Captures a fenced python block. DOTALL so we can span newlines. The closing
# triple-backtick is optional (the model sometimes runs out of tokens mid-block).
_PY_BLOCK_RE = re.compile(r"```python\s*(.*?)(?:```|$)", re.DOTALL | re.IGNORECASE)


def tool_augmented(question):
    """Ask the model to write a Python snippet that PRINTS the answer, run it,
    then ask the model to read the output and state the final answer.

    Two calls. If `math.<something>` is referenced but `import math` is missing,
    we inject the import - the model forgets it more often than you'd expect.
    """
    if not CALLS.can_afford(2):
        return cot(question)

    # 1) Write the snippet.
    write_prompt = (
        "Write a short Python snippet that PRINTS the answer to the problem. "
        "Only the `math` module is allowed. Wrap the code in a fenced block like:\n"
        "```python\n"
        "# your code here\n"
        "print(result)\n"
        "```\n\n"
        "Problem:\n" + question
    )
    snippet_reply = ask(write_prompt, temperature=0.0, max_tokens=512)

    m = _PY_BLOCK_RE.search(snippet_reply)
    if m:
        code = m.group(1).strip()
    else:
        # No fenced block parsed - fall back to treating the whole reply as code.
        code = snippet_reply.strip()

    # Robustness fix: if the snippet uses math.<x> but forgot the import, add it.
    if "math." in code and "import math" not in code:
        code = "import math\n" + code

    output = python_exec(code)

    # 2) Interpret the output.
    interpret_prompt = (
        "Problem:\n" + question
        + "\n\nA Python snippet produced this output:\n" + (output or "(no output)")
        + "\n\nWhat is the final answer to the problem?"
        + FINAL_ANSWER_INSTRUCTION
    )
    final_reply = ask(interpret_prompt, temperature=0.0, max_tokens=128)
    return extract_final_answer(final_reply)


In [22]:
def plan_and_solve(question):
    """Two calls: (1) write a 3-5 step numbered plan, (2) execute the plan."""
    if not CALLS.can_afford(2):
        return cot(question)

    plan_prompt = (
        "Write a 3-5 step numbered plan for solving this problem. Only the plan - "
        "do NOT solve it or state the answer yet.\n\nProblem:\n" + question
    )
    plan = ask(plan_prompt, temperature=0.0, max_tokens=256)

    solve_prompt = (
        "Problem:\n" + question
        + "\n\nPlan:\n" + plan
        + "\n\nExecute the plan step by step and state the final answer."
        + FINAL_ANSWER_INSTRUCTION
    )
    reply = ask(solve_prompt, temperature=0.0, max_tokens=512)
    return extract_final_answer(reply)


## 9) Domain-aware agent

The `Agent` picks a technique based on the instance's domain (from the dataset, or from `infer_domain` as a 1-call fallback). The router is an explicit chain of `if` branches - not a hardcoded `tool(question)` delegation - so every technique above remains selectable and the dispatch logic is visible in one place.


In [23]:
_DOMAINS = ["math", "common_sense", "coding", "planning", "future_prediction"]


def infer_domain(question):
    """One-call domain classifier. Used when instance['domain'] is missing."""
    # If we can't afford even the classifier call, pick a safe default and let the
    # agent proceed. common_sense routes to self_consistency, which works broadly.
    if not CALLS.can_afford(1):
        return "common_sense"

    prompt = (
        "Classify the following question into exactly ONE of these domains:\n"
        "- math\n"
        "- common_sense\n"
        "- coding\n"
        "- planning\n"
        "- future_prediction\n\n"
        "Reply with only the domain name, nothing else.\n\n"
        "Question:\n" + question
    )
    reply = ask(prompt, temperature=0.0, max_tokens=16).strip().lower()
    # Match the first domain name that appears in the reply.
    for d in _DOMAINS:
        if d in reply:
            return d
    return "common_sense"


class Agent:
    """Domain-aware router over the eight techniques.

    Routing:
      math              -> tool_augmented, retry with least_to_most if the result
                           has no digit and budget allows
      coding            -> react, polish with self_refine if budget allows
      common_sense      -> self_consistency
      planning          -> plan_and_solve; upgrade to tree_of_thoughts if we've
                           barely used the budget and can afford 10 more calls
      future_prediction -> plan_and_solve, then self_refine if budget allows
      else              -> cot (fallback)

    The router is a chain of `if` branches - explicit per the assignment's
    no-hardcoded-delegation rule - so every technique remains selectable.
    """

    def __init__(self, hard_limit=20):
        self.hard_limit = hard_limit
        # Keep the global tracker in sync with whatever limit this Agent was built with.
        CALLS.hard_limit = hard_limit

    def solve(self, instance):
        CALLS.reset()

        question = instance["input"]
        # Use the dataset's domain label when present; otherwise classify ourselves.
        domain = instance.get("domain") or infer_domain(question)

        technique_used = ""
        answer = ""

        if domain == "math":
            technique_used = "tool_augmented"
            answer = tool_augmented(question)
            # If the answer has no digits, the Python snippet likely failed; retry
            # with least_to_most (decompose the word problem) if we can afford it.
            if not re.search(r"\d", answer or "") and CALLS.can_afford(5):
                retry = least_to_most(question)
                if retry:
                    technique_used = "tool_augmented+least_to_most"
                    answer = retry

        elif domain == "coding":
            technique_used = "react"
            answer = react(question)
            # self_refine costs up to 3 calls. If we have room, run it to catch
            # first-pass bugs (wrong indexing, off-by-one, etc.).
            if answer and CALLS.can_afford(3):
                polished = self_refine(question)
                if polished:
                    technique_used = "react+self_refine"
                    answer = polished

        elif domain == "common_sense":
            technique_used = "self_consistency"
            answer = self_consistency(question, n_samples=5)

        elif domain == "planning":
            # Start cheap; if we've used very little budget (the infer_domain call
            # might have consumed 1) we upgrade to the deeper ToT search.
            technique_used = "plan_and_solve"
            answer = plan_and_solve(question)
            if CALLS.used <= 2 and CALLS.can_afford(10):
                upgraded = tree_of_thoughts(question)
                if upgraded:
                    technique_used = "tree_of_thoughts"
                    answer = upgraded

        elif domain == "future_prediction":
            technique_used = "plan_and_solve"
            answer = plan_and_solve(question)
            if answer and CALLS.can_afford(3):
                polished = self_refine(question)
                if polished:
                    technique_used = "plan_and_solve+self_refine"
                    answer = polished

        else:
            technique_used = "cot"
            answer = cot(question)

        return {
            "answer": answer or "",
            "technique": technique_used,
            "domain": domain,
            "calls_used": CALLS.used,
        }


## 10) Evaluation loop

Cheap-first grading: try normalized equality and numeric extraction before paying for the LLM judge. Only fall back to the template's `self_evaluate` when cheap grading fails - this avoids burning an extra ~1,000 calls on the full dev run.


In [26]:
# Alias the template's normalize_text under the name used below, so we reuse the
# existing helper instead of duplicating it.
normalize_answer = normalize_text


def _cheap_grade(expected, got):
    """Return True iff normalized strings match, or extracted numbers match."""
    if got is None or got == "":
        return False
    # Text-normalized equality first (handles 'second' vs 'second.' vs 'Second').
    if normalize_answer(expected) == normalize_answer(got):
        return True
    # Numeric equality (handles '8' vs '8.0' and answers wrapped like 'Answer: 8').
    exp_num = extract_number(expected)
    got_num = extract_number(got)
    if exp_num is not None and got_num is not None:
        try:
            if float(exp_num) == float(got_num):
                return True
        except ValueError:
            pass
    return False


def run_eval(agent, data, limit=None, verbose=True, use_judge_fallback=True):
    """Run the agent over the dataset, report stats, and return per-instance rows.

    Grading is cheap-first: normalized equality / numeric match runs for free, and
    only when that misses do we spend an LLM call on the template's self_evaluate
    judge. This matters a lot at 1,000 instances - without the cheap path the
    judge would cost us another ~1,000 calls.
    """
    if limit is not None:
        data = data[:limit]

    results = []
    for idx, inst in enumerate(data):
        expected = inst.get("output", "")
        out = agent.solve(inst)

        # Pass 1: cheap grading.
        correct = _cheap_grade(expected, out["answer"])
        judged_by = "cheap"

        # Pass 2: LLM judge only when cheap grading said no.
        if not correct and use_judge_fallback:
            try:
                correct = bool(self_evaluate(
                    question=inst["input"],
                    prediction=out["answer"],
                    expected_answer=expected,
                ))
                judged_by = "judge"
            except Exception as e:
                # Don't let a flaky judge call tank the whole run - record and move on.
                judged_by = f"judge_error:{type(e).__name__}"

        row = {
            "index": idx,
            "domain": out["domain"],
            "technique": out["technique"],
            "expected": expected,
            "answer": out["answer"],
            "correct": bool(correct),
            "judged_by": judged_by,
            "calls_used": out["calls_used"],
        }
        results.append(row)

        # Print progress for the first 5 rows and every 25th after that.
        if verbose and (idx < 5 or (idx + 1) % 25 == 0):
            mark = "OK" if correct else "NO"
            preview = (out["answer"] or "")[:40]
            print(
                f"[{idx+1}/{len(data)}] {mark}  "
                f"domain={row['domain']:<17}  "
                f"tech={row['technique']:<32}  "
                f"calls={row['calls_used']:<2}  "
                f"got={preview!r}"
            )

    # Summary stats.
    total = len(results)
    correct_count = sum(1 for r in results if r["correct"])
    total_calls = sum(r["calls_used"] for r in results)
    avg_calls = (total_calls / total) if total else 0.0
    violations = sum(1 for r in results if r["calls_used"] > 20)

    print("\n=== Summary ===")
    pct = (100.0 * correct_count / total) if total else 0.0
    print(f"Accuracy: {correct_count}/{total} = {pct:.1f}%")
    print(f"Avg calls/question: {avg_calls:.2f}")
    print(f"Budget violations (>20 calls): {violations}")

    # Per-domain breakdown.
    by_domain = {}
    for r in results:
        d = r["domain"]
        if d not in by_domain:
            by_domain[d] = {"n": 0, "c": 0, "calls": 0}
        by_domain[d]["n"] += 1
        by_domain[d]["c"] += int(r["correct"])
        by_domain[d]["calls"] += r["calls_used"]

    print("\nPer-domain:")
    for d, stats in sorted(by_domain.items(), key=lambda kv: -kv[1]["n"]):
        acc = 100.0 * stats["c"] / stats["n"]
        avg = stats["calls"] / stats["n"]
        print(f"  {d:<20} n={stats['n']:<4}  acc={acc:5.1f}%  avg_calls={avg:.2f}")

    # Save full per-instance results for later analysis / the report.
    try:
        with open("dev_results.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        print("\nSaved per-instance results -> dev_results.json")
    except Exception as e:
        print(f"(Warning: could not save dev_results.json: {e})")

    return results


## 11) Runs

A 10-instance smoke test runs immediately when you execute the notebook. The full 1,000-instance run is kept commented out below so the notebook is cheap to reopen.


In [27]:
# Smoke test: 10 instances from the dev set. Runs on notebook execute.
agent = Agent(hard_limit=20)
rows_smoke = run_eval(agent, DEV_DATA, limit=10)


[1/10] NO  domain=math               tech=tool_augmented+least_to_most      calls=7   got='140'
[2/10] OK  domain=math               tech=tool_augmented+least_to_most      calls=7   got='164'
[3/10] NO  domain=math               tech=tool_augmented                    calls=2   got='x^2 + 18x + 45 = (x^2'
[4/10] NO  domain=math               tech=tool_augmented+least_to_most      calls=7   got='144.5'
[5/10] NO  domain=math               tech=tool_augmented+least_to_most      calls=7   got='- Units digit must be even and different'

=== Summary ===
Accuracy: 2/10 = 20.0%
Avg calls/question: 6.00
Budget violations (>20 calls): 0

Per-domain:
  math                 n=10    acc= 20.0%  avg_calls=6.00

Saved per-instance results -> dev_results.json


In [28]:
# Full dev run - all 1,000 instances. Uncomment to execute; this takes a while.
#
# agent = Agent(hard_limit=20)
# rows_full = run_eval(agent, DEV_DATA, limit=None)


## Notes for the report

Include the following in the 1-page writeup:

1. **Agent overview** - one paragraph describing the budget tracker (`CallBudget`) plus the domain-aware router (`Agent.solve`).
2. **The eight inference-time techniques (with cell pointers)** - reference each by its cell in Section 8:
   - `cot` - zero-shot chain-of-thought baseline (1 call).
   - `self_consistency` - 5 samples, majority vote.
   - `tree_of_thoughts` - breadth-first generate + score + expand.
   - `self_refine` - draft > critique > revise, with short-circuit on "NO ISSUES".
   - `react` - Thought/Action loop over `python_exec`.
   - `least_to_most` - decompose, solve serially, synthesize.
   - `tool_augmented` - model writes Python, we run it, model interprets output.
   - `plan_and_solve` - numbered plan, then execute.
3. **Budget strategy** - the hard 20-call limit enforced by `CallBudget`; every technique begins with `CALLS.can_afford(n)` and falls back to `cot()` when the budget is too tight.
4. **Routing policy table** - one row per domain with {domain, primary technique, upgrade condition, fallback}.
5. **Dev-set results** - overall accuracy from `run_eval`, per-domain accuracy, avg calls/question, count of budget violations (should be 0).
6. **Known failure modes** - anything surfaced by the smoke run or full run (e.g., math answers that come back non-numeric, ReAct loops that exhaust `max_steps`).
